# Lab 2: Document Structure

Read the sentence pairs of the *SMTeuroparl* test set, compute their similarity using words and the Jaccard distance, and compare it with the gold standard using the Pearson correlation.

In [1]:
from pathlib import Path

# The notebook runs from labs/, so the workspace root is the parent folder.
ROOT = Path.cwd().parent

DATA_DIR = ROOT / "data" / "test_gold"
NLTK_DIR = ROOT / "models" / "nltk_data"

import pandas as pd
import nltk
from nltk.metrics.distance import jaccard_distance
from scipy.stats import pearsonr

# NLTK packages needed by this notebook.
nltk.data.path.insert(0, str(NLTK_DIR))
nltk.download("punkt_tab", download_dir=NLTK_DIR)

[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/Ran/Projects/UPC/IHLT/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [2]:
# Load sentence pairs and gold-standard similarity scores
dt = pd.read_csv(DATA_DIR / 'STS.input.SMTeuroparl.txt', sep='\t', header=None)
dt['gs'] = pd.read_csv(DATA_DIR / 'STS.gs.SMTeuroparl.txt', sep='\t', header=None)
print(f"Sentence pairs: {len(dt)}")

Sentence pairs: 459


In [3]:
jaccards = []

# Compute a Jaccard similarity score for each pair of sentences.
for i in range(len(dt)):
    s1 = dt.iloc[i, 0]
    s2 = dt.iloc[i, 1]
    words1 = set(nltk.word_tokenize(s1))
    words2 = set(nltk.word_tokenize(s2))

    # NLTK returns Jaccard distance, so convert it to similarity.
    # Similarity = 1 - distance.
    j_distance = jaccard_distance(words1, words2)
    j_similarity = 1 - j_distance
    jaccards.append(j_similarity)

dt['jaccard'] = jaccards

# The vector of similarities, next to the gold standard.
dt[['gs', 'jaccard']].head()

,gs,jaccard
0,4.50,0.346154
1,5.00,0.785714
2,4.25,0.391304
3,4.50,0.545455
4,5.00,1.000000


In [4]:
# Compare our Jaccard similarity scores with the gold-standard scores
pearson = pearsonr(dt['gs'], dt['jaccard'])[0]
print(f"Pearson correlation: {pearson:.4f}")

Pearson correlation: 0.4507


In [5]:
# Distribution of the gold standard and of the Jaccard similarities.
print(dt[['gs', 'jaccard']].describe().round(3))
print(f"\nPairs with gold score >= 4: {(dt['gs'] >= 4).mean():.1%}")

# Mean Jaccard similarity for each range of gold scores.
gold_range = pd.cut(dt['gs'], [0, 3, 4, 4.5, 5], include_lowest=True)
dt.groupby(gold_range, observed=True)['jaccard'].agg(['count', 'mean', 'min', 'max']).round(3)

            gs  jaccard
count  459.000  459.000
mean     4.550    0.500
std      0.516    0.243
min      1.500    0.053
25%      4.250    0.319
50%      4.750    0.435
75%      5.000    0.600
max      5.000    1.000

Pairs with gold score >= 4: 90.6%


,count,mean,min,max
gs,,,,
"(-0.001, 3.0]",13,0.374,0.095,0.538
"(3.0, 4.0]",61,0.342,0.111,0.750
"(4.0, 4.5]",123,0.395,0.053,0.900
"(4.5, 5.0]",262,0.593,0.182,1.000


In [6]:
def show_pair(i):
    """Print a sentence pair with its gold score, Jaccard similarity and word sets."""
    s1, s2 = dt.iloc[i, 0], dt.iloc[i, 1]
    words1, words2 = set(nltk.word_tokenize(s1)), set(nltk.word_tokenize(s2))
    print(f"Pair {i}  (gold {dt['gs'][i]}, Jaccard {dt['jaccard'][i]:.3f})")
    print(f"  S1: {s1}")
    print(f"  S2: {s2}")
    print(f"  shared:  {sorted(words1 & words2)}")
    print(f"  only S1: {sorted(words1 - words2)}")
    print(f"  only S2: {sorted(words2 - words1)}")
    print()

# Paraphrases with the maximum gold score but low word overlap.
print("=== Gold 5, lowest Jaccard ===\n")
for i in dt[dt['gs'] == 5].nsmallest(2, 'jaccard').index:
    show_pair(i)

# Pairs with a low gold score but a relatively high word overlap.
print("=== Gold < 3, highest Jaccard ===\n")
for i in dt[dt['gs'] < 3].nlargest(2, 'jaccard').index:
    show_pair(i)

# Pair 12: same meaning, but different case and inflection.
print("=== Case and inflection ===\n")
show_pair(12)

=== Gold 5, lowest Jaccard ===

Pair 382  (gold 5.0, Jaccard 0.235)
  S1: That provision could open the door wide to arbitrariness.
  S2: This point of the regulations opens the door to the arbitrary one.
  shared:  ['.', 'door', 'the', 'to']
  only S1: ['That', 'arbitrariness', 'could', 'open', 'provision', 'wide']
  only S2: ['This', 'arbitrary', 'of', 'one', 'opens', 'point', 'regulations']

Pair 385  (gold 5.0, Jaccard 0.235)
  S1: That provision could open the door wide to arbitrariness.
  S2: This point of the regulations opens the door to arbitrary power.
  shared:  ['.', 'door', 'the', 'to']
  only S1: ['That', 'arbitrariness', 'could', 'open', 'provision', 'wide']
  only S2: ['This', 'arbitrary', 'of', 'opens', 'point', 'power', 'regulations']

=== Gold < 3, highest Jaccard ===

Pair 392  (gold 2.5, Jaccard 0.444)
  S1: Thank you, Commissioner.
  S2: Thank you, Mister the Police chief.
  shared:  [',', '.', 'Thank', 'you']
  only S1: ['Commissioner']
  only S2: ['Mister', 'Pol

## Justification

The Pearson correlation between the Jaccard similarities and the gold standard is **0.4507**: a positive but moderate correlation. Word overlap captures part of the similarity, but far from all of it.

**Why it is positive.** Pairs with more words in common tend to have higher gold scores. The pairs with a gold score above 4.5 have a mean Jaccard similarity of 0.593, against 0.342–0.395 for the other ranges, and pair 4, where both sentences have the same set of words, gets both a gold score of 5 and a Jaccard similarity of 1.

**Why it is only moderate.**

- **Paraphrases with different words.** Pair 382 has the maximum gold score (5.0), but a Jaccard similarity of only 0.235: *provision* / *point of the regulations*, *open* / *opens* and *arbitrariness* / *arbitrary* mean the same, but they are different words, and only *door*, *the*, *to* and *.* are shared.
- **Surface variants count as different words.** The comparison is case-sensitive and uses inflected forms. In pair 12 (gold 4.75), *the* / *The* and *adopted* / *adopts* are counted as different words, so the similarity is only 0.556.
- **Function words and punctuation count as much as content words.** In pair 392 (gold 2.5, Jaccard 0.444), the shared words are only *Thank*, *you*, *,* and *.*, while the words that carry the meaning (*Commissioner* vs *Police chief*) are different.
- **The gold standard is concentrated at the top of the scale.** The mean gold score is 4.55 and 90.6% of the pairs have a score of 4 or more (standard deviation 0.516), while the Jaccard similarity spreads from 0.053 to 1.0. Most pairs are near-paraphrases with very similar gold scores but very different word overlaps, which limits the correlation. The relation is not even monotonic at the bottom: the pairs with a gold score of 3 or less have a higher mean Jaccard similarity (0.374) than those between 3 and 4 (0.342).

So the Jaccard similarity of words is a reasonable baseline, but it only measures surface overlap. Normalizing the words (lowercasing, lemmatization), ignoring stop words and punctuation, or using lexical resources for synonyms could improve the correlation.